**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/alonso-2020"], check=True)
    os.chdir("/content/encore/entries/alonso-2020")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Alonso & Marder (2020), temperature ramps, from Dryad m31b1h7

Paper: Alonso LM, Marder E (2020), eLife 9:e55470, doi:10.7554/eLife.55470; claims in
`claims/alonso-2020.yaml` (version of record, main text, legends and Methods).

Dataset: Dryad doi:10.5061/dryad.m31b1h7, version id 94811 (the only published version, so the
one pinned), file `alonso_marder_tempcomp_2019_root.zip` (MD5 from Dryad's file list checked on
load; SHA-256 printed below). The models are simulated with the port in `pyloric.py`; the main
notebook, `reproduce_alonso_2020.ipynb`, compares it with the deposited Java and checks the data.

Two ramp protocols:

- up: 10 to 25 degrees C over 60 min (Results, Duty cycle and phase maintenance, paragraph 5,
  "ramps from 10 C to 25 C over 60 min"), model PLKTEM, for figs2-ramp-largest-isi;
- updown: 10 to 35 degrees C in 30 min and back to 10 degrees C at the same rate (Results,
  Hysteresis and multistability, paragraph 1), every deposited model (38), for
  fig8a-hysteresis-quiescence, fig8-up-down-ramps and fig8b-pathological-attractor.

The second ramp reading of fig8-up-down-ramps (10 to 25 degrees C over 60 min and back, every
model) is not run: the cell timed in the fig8-up-down-ramps section estimates it from a subset.
The up ramp of every model, the main reading of figs3-duty-cycle-variable-across-models, is not
run either (its estimate is printed in the same cell); the main notebook tests that claim on its
variant, the 16 fixed temperatures.

The deposit holds no ramp code; the temperature changes linearly in time, the temperature
factors are updated every 20 steps (1 ms) at the temperature of the middle of those steps, and
each ramp starts from the deposited initial conditions at its first temperature (assumptions;
`pyloric.py`).

| Claim | Tested |
|---|---|
| figs2-ramp-largest-isi | tested |
| fig8a-hysteresis-quiescence | tested |
| fig8-up-down-ramps | blocked (too_large); the first ramp reading is shown, the second timed |
| fig8b-pathological-attractor | tested |

In [ ]:
import resource
import time
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import pyloric as P

T_START = time.time()
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)
raw = P.read_zip_bytes()
print(P.digests(raw))
Z = P.open_zip(raw)
MODELS = P.load_models(Z)
CI = P.load_ci(Z)
NAMES = list(MODELS)

## Data check and the runs

The runs below take most of this notebook's time; their run time is printed.

In [ ]:
print(len(NAMES), "models:", NAMES)
print("parameters per model:", {len(p) for p in MODELS.values()}, "; initial state:", len(CI), "numbers")
t0 = time.time()
tasks = [("UP PLKTEM", MODELS["PLKTEM"], CI, [(10.0, 25.0, 3600.0)])]
tasks += [(n, MODELS[n], CI, [(10.0, 35.0, 1800.0), (35.0, 10.0, 1800.0)]) for n in NAMES]
out = P.pmap(P.run_ramp, tasks)
UP = {"PLKTEM": out[0]}
UD = {r["key"]: r for r in out[1:]}
print(f"ramps: {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print("runs that diverged (NaN):", [n for n in UP if not UP[n]["legs"][0]["finite"]],
      [(n, i) for n in NAMES for i in range(2) if not UD[n]["legs"][i]["finite"]])
print("PLKTEM up ramp, spikes per cell:", [len(s) for s in UP["PLKTEM"]["legs"][0]["spikes"]])
spk = pd.DataFrame({n: [len(s) for leg in UD[n]["legs"] for s in leg["spikes"]] for n in NAMES},
                   index=["up PD", "up LP", "up PY", "down PD", "down LP", "down PY"]).T
print("spike counts per run and cell:")
print(spk.describe().round(0).to_string())

## figs2-ramp-largest-isi

> During a temperature ramp from 10 to 25 degrees C over 60 min, the spiking patterns change in
> complicated ways and the largest ISI decreases monotonically, in step with the rising burst
> frequency.

PLKTEM, the up ramp. The largest ISI of each burst cycle (burst start to the next); per window,
its median over the cycles that start in the window. Readings: windows of 1 degree C (4 min) and
of 1 min.

In [ ]:
leg = UP["PLKTEM"]["legs"][0]
res2 = {}
for label, w in (("a, 4 min", 240.0), ("b, 1 min", 60.0)):
    edges = np.arange(0, 3600.0 + 1e-9, w)
    temps = 10 + 15 * (edges[:-1] + w / 2) / 3600
    out = {}
    for c, cell in enumerate(P.CELLS):
        t, isi = P.cycle_largest_isi(leg["spikes"][c])
        med = np.array([np.median(isi[(t >= a) & (t < b)]) if ((t >= a) & (t < b)).any() else np.nan for a, b in zip(edges[:-1], edges[1:])])
        ok = np.isfinite(med)
        rho = stats.spearmanr(temps[ok], med[ok])[0]
        inc = np.where(np.diff(med[ok]) > 0)[0]
        out[cell] = dict(rho=rho, increases=len(inc), windows=int(ok.sum()), first=med[ok][0], last=med[ok][-1])
        if label.startswith("a"):
            print(cell, "median largest ISI per window (s):", np.round(med, 3))
    res2[label] = out
    print(f"reading ({label}):", {k: {kk: (round(vv, 3) if isinstance(vv, float) else vv) for kk, vv in v.items()} for k, v in out.items()})
    print("   cells with Spearman below -0.9:", sum(v["rho"] < -0.9 for v in out.values()))
fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for c, cell in enumerate(P.CELLS):
    t, isi = P.cycle_largest_isi(leg["spikes"][c])
    ax[c].plot(10 + 15 * t / 3600, isi, ".", ms=1); ax[c].set_title(cell); ax[c].set_xlabel("temperature (C)")
    print(f"{cell}: per-cycle largest ISI from {isi.min():.3f} to {isi.max():.3f} s, {len(isi)} cycles")
ax[0].set_ylabel("largest ISI per cycle (s)"); plt.tight_layout(); plt.show()

**Result: reproduced.** On the 60 min ramp of PLKTEM the median per-cycle largest ISI falls in
every 4 min window, with no increase, in all three cells (PD 0.604 to 0.299 s, LP 0.613 to 0.301 s,
PY 0.522 to 0.251 s; Spearman -1.000 each). In 1 min windows the Spearman correlations are -0.998,
-0.998 and -0.994, with 7, 9 and 22 increases between windows.

## fig8a-hysteresis-quiescence

> During a temperature ramp from 10 to 35 degrees C and back, the PD cell of one model becomes
> quiescent before reaching 35 degrees C and resumes spiking on the way down at a temperature
> about 2 degrees C lower than where it stopped ...

OSHNMD of the Methods is 0SHNMD in the deposit (the letter O read as the digit 0); H5AIZR is the
variant.

In [ ]:
def fig8a(name):
    up, down = UD[name]["legs"]
    su, sd = up["spikes"][0], down["spikes"][0]
    if len(su) == 0 or len(sd) == 0:
        print(name, "no PD spikes on a leg"); return None
    t_last = su[-1]; T_last = 10 + 25 * t_last / 1800
    t_first = sd[0]; T_first = 35 - 25 * t_first / 1800
    gap_up = 1800 - t_last
    print(f"{name}: last PD spike on the up ramp at {t_last:.1f} s ({T_last:.2f} C); first PD spike on the down ramp at "
          f"{t_first:.1f} s after its start ({T_first:.2f} C); quiescent for {gap_up + t_first:.1f} s across the top; "
          f"difference {T_last - T_first:.2f} C")
    lu = np.diff(su); print(f"   longest PD ISI on the up ramp before the last spike: {lu.max():.2f} s")
    return T_last, T_first, T_last - T_first, t_last

r8a = fig8a("0SHNMD")
r8a_var = fig8a("H5AIZR")
for n in NAMES:
    up, down = UD[n]["legs"]
    if len(up["spikes"][0]) and up["spikes"][0][-1] < 1790:
        print(f"   models whose PD stops before 35 C on the up ramp: {n}, last spike at {10 + 25 * up['spikes'][0][-1] / 1800:.2f} C")

**Result: reproduced.** On the updown ramp, 0SHNMD's PD fires its last spike on the up ramp at
34.16 C, stays quiescent across the top for 260 s and resumes at 32.22 C on the down ramp, 1.94 C
lower (2 +- 0.2). The last spike comes at 1740 s, not before 600 s as the legend says (not judged).
The variant H5AIZR differs by 0.05 C.

## fig8-up-down-ramps

> In 34 of the 36 models the spiking patterns over the working range are similar but not
> identical on the up and down ramps ...

The updown ramp; per cell and per 1 degree window from 10 to 25 degrees C (72 s each), mean duty
cycle, mean burst frequency and median per-cycle largest ISI on each leg; similar when all are
within 10% between legs in every window and the network is triphasic in every window on both.
The second ramp reading (10 to 25 degrees C over 60 min and back) is not run; the cell after the
comparison times it on a subset.

In [ ]:
def legs_compare(name):
    up, down = UD[name]["legs"]
    worst, trip, ident = 0.0, True, True
    for k in range(15):
        a, b = 72.0 * k, 72.0 * (k + 1)                   # up: 10+k to 11+k C
        da, db = 1800 - 72.0 * (k + 1), 1800 - 72.0 * k     # down: the same temperatures
        mu, tu = P.window_measures(up["spikes"], a, b)
        md, td = P.window_measures(down["spikes"], da, db)
        rel = np.abs(md / mu - 1)
        worst = max(worst, np.nanmax(rel) if np.isfinite(rel).any() else np.inf)
        if not np.isfinite(rel).all():
            worst = np.inf
        trip = trip and tu and td
    nu = [len(s[s < 1080]) for s in up["spikes"]]
    nd = [len(s[s >= 720]) for s in down["spikes"]]
    return worst, trip, nu, nd

rows = []
for n in NAMES:
    worst, trip, nu, nd = legs_compare(n)
    rows.append(dict(model=n, largest_rel_diff=worst, triphasic_all_windows=trip, similar=bool(worst <= 0.1 and trip),
                     spikes_up=sum(nu), spikes_down=sum(nd)))
t8 = pd.DataFrame(rows).set_index("model")
print(t8.round(3).to_string())
print(f"\nsimilar (within 10% and triphasic in every window): {int(t8.similar.sum())} of {len(t8)}")
print(f"identical spike counts on the two legs (10 to 25 C): {int((t8.spikes_up == t8.spikes_down).sum())}")
for thr in (0.05, 0.1, 0.2):
    print(f"   threshold {thr:.2f}: similar {int(((t8.largest_rel_diff <= thr) & t8.triphasic_all_windows).sum())}")

In [ ]:
# The second ramp reading (10 to 25 C over 60 min and back, 7200 s per model) timed on a subset:
# four models, 120 s of each leg, run as the full reading would be (4 processes).
t0 = time.time()
sub = P.pmap(P.run_ramp, [(n, MODELS[n], CI, [(10.0, 10.5, 120.0), (10.5, 10.0, 120.0)]) for n in NAMES[:4]])
rate = 4 * 240.0 / (time.time() - t0)
print(f"subset: {rate:.0f} simulated seconds per second on 4 processes")
print(f"estimate for the second reading, 38 models x 7200 s: {38 * 7200 / rate / 60:.0f} min; "
      f"for the up legs alone (the main reading of figs3-duty-cycle-variable-across-models), 38 x 3600 s: {38 * 3600 / rate / 60:.0f} min")
print(f"this notebook has run for {(time.time() - T_START) / 60:.0f} min so far; the limit for a notebook is 60 min")

**Result: blocked (too_large).** On the first ramp reading (10 to 35 C and back over 60 min), 22
of 38 models are similar on the two legs (within 10% and triphasic in every 1 degree window; 9 at
5%, 28 at 20%), outside 34 +- 3, so this reading does not meet the criterion. Six models diverge
numerically on this ramp, MAXTTP's cells fire no spike on the down leg below 25 C, and 9 others
exceed the 10% bound or lose triphasic order. No model has identical spike counts on the two legs.
The criterion is met when either reading gives a count within tolerance, and the second reading
(10 to 25 C over 60 min and back) was not run: the estimate printed above, from a subset, is
well over the 60 min a notebook may take, on its own as well as beside the runs of this notebook.

## fig8b-pathological-attractor

> In a second model hysteresis is marked; at high temperatures it produces ISIs as long as about
> 1 s while spiking at all temperatures, and its PD cell passes from regular bursting into a
> pathological state that persists on the down ramp until about 20 degrees C.

H5AIZR (0SHNMD the variant). PD classified in successive 1 min windows of the updown ramp.

In [ ]:
def fig8b(name):
    up, down = UD[name]["legs"]
    spikes = np.r_[up["spikes"][0], 1800 + down["spikes"][0]]
    swu = np.r_[up["sw_up"][0], 1800 + down["sw_up"][0]]
    cls, temps = [], []
    for k in range(60):
        a, b = 60.0 * k, 60.0 * (k + 1)
        s = spikes[(spikes >= a) & (spikes < b)] - a
        u = swu[(swu >= a) & (swu < b)] - a
        cls.append(P.classify_cell(s, u))
        mid = a + 30
        temps.append(10 + 25 * mid / 1800 if mid < 1800 else 35 - 25 * (mid - 1800) / 1800)
    cls = np.array(cls); temps = np.array(temps)
    nspk = [int(((spikes >= 60 * k) & (spikes < 60 * (k + 1))).sum()) for k in range(60)]
    rb = cls == "regular bursting"
    left_up = not rb[:30].all()
    # return: the start of the last run of regular bursting windows on the down ramp
    k = 59
    while k >= 30 and rb[k]:
        k -= 1
    T_return = temps[k + 1] if k < 59 else np.nan
    sel = (spikes >= 1500) & (spikes < 2100)
    longest = np.diff(spikes[sel]).max() if sel.sum() > 1 else np.nan
    print(f"{name}: PD classes per minute:")
    for i in range(0, 60, 6):
        print("   ", [f"{temps[j]:.1f}C {cls[j]}" for j in range(i, i + 6)])
    print(f"   spikes in every window {min(nspk) > 0} (fewest {min(nspk)}); regular bursting in the first window {rb[0]}; "
          f"leaves regular bursting on the up ramp {left_up}; returns to it for good on the down ramp at {T_return:.2f} C; "
          f"longest PD ISI between 25 and 35 min {longest:.3f} s")
    return dict(all_spiking=min(nspk) > 0, early_rb=bool(rb[0]), left=left_up, T_return=T_return, longest=longest)

r8b = fig8b("H5AIZR")
r8b_var = fig8b("0SHNMD")

**Result: not reproduced.** H5AIZR's PD spikes in every 1 min window, bursts regularly early on
the up ramp and leaves regular bursting at 28.8 C, but it is regular again at 32 to 35 C and on the
down ramp it returns to regular bursting for good at 17.08 C (20 +- 2), after isolated irregular
windows at 27.1, 22.1 and 17.9 C; its longest PD ISI between 25 and 35 min is 11.32 s (1 +- 0.1).
The variant 0SHNMD has PD windows with no spike.

## Summary

| Claim | Outcome |
|---|---|
| figs2-ramp-largest-isi | reproduced |
| fig8a-hysteresis-quiescence | reproduced |
| fig8-up-down-ramps | blocked (too_large; first reading 22 of 38, second reading estimated over 60 min) |
| fig8b-pathological-attractor | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the deposit has no ramp code, so the temperature is
changed linearly and the factors updated every 20 steps (1 ms); each ramp starts from the
deposited initial conditions. The second ramp reading of fig8-up-down-ramps was not run (the
estimate from a subset is printed in its section), nor was the 60 min ramp of every model for
figs3-duty-cycle-variable-across-models (estimated in the same cell). On the updown ramp the deposited integrator diverges (NaN) for G7P2WE,
INPO74, J9U8SQ, NJP4RT, R00977 and SJR46Y.

In [ ]:
print(f"notebook run time {(time.time() - T_START) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")